In [1]:
import pandas as pd

In [1]:
"""
Extract an Esri ArcGIS Online Feature Layer to CSV (lat, lon + all attributes).

Usage:
    python extract_feature_layer.py <feature_layer_url> [output.csv]

Example URLs:
    https://services.arcgis.com/.../FeatureServer/0
    https://services.arcgis.com/.../MapServer/0
"""

import sys
import csv
import json
import urllib.request
import urllib.parse

# ── Configuration ─────────────────────────────────────────────────────────────
BATCH_SIZE = 1000          # records per request (ArcGIS default max is 1000–2000)
OUTPUT_WKT = 4326          # WGS-84 (lat/lon); do not change unless you know why
# ──────────────────────────────────────────────────────────────────────────────


def query_layer(base_url: str, offset: int, batch: int) -> dict:
    """Fetch one page of features from the REST endpoint."""
    params = urllib.parse.urlencode({
        "where": "1=1",
        "outFields": "*",
        "outSR": OUTPUT_WKT,
        "returnGeometry": "true",
        "resultOffset": offset,
        "resultRecordCount": batch,
        "f": "json",
    })
    url = f"{base_url.rstrip('/')}/query?{params}"
    with urllib.request.urlopen(url, timeout=60) as resp:
        return json.loads(resp.read().decode())


def get_layer_info(base_url: str) -> dict:
    """Fetch layer metadata (name, geometry type, max record count)."""
    params = urllib.parse.urlencode({"f": "json"})
    url = f"{base_url.rstrip('/')}?{params}"
    with urllib.request.urlopen(url, timeout=30) as resp:
        return json.loads(resp.read().decode())


def extract_coords(geometry: dict | None) -> tuple[float | None, float | None]:
    """
    Return (latitude, longitude) from an Esri geometry dict.
    Handles point geometries; for lines/polygons uses the first vertex.
    """
    if not geometry:
        return None, None

    # Point
    if "x" in geometry and "y" in geometry:
        return geometry["y"], geometry["x"]

    # Polyline / Polygon – use first vertex of first ring/path
    for key in ("rings", "paths"):
        if key in geometry and geometry[key]:
            pt = geometry[key][0][0]
            return pt[1], pt[0]

    return None, None


def main(base_url: str, output_path: str) -> None:
    # ── Layer metadata ────────────────────────────────────────────────────────
    print(f"Connecting to: {base_url}")
    info = get_layer_info(base_url)

    if "error" in info:
        sys.exit(f"Layer error: {info['error'].get('message', info['error'])}")

    layer_name = info.get("name", "layer")
    max_count  = info.get("maxRecordCount", BATCH_SIZE)
    batch      = min(BATCH_SIZE, max_count)
    geom_type  = info.get("geometryType", "unknown")
    print(f"Layer: '{layer_name}'  |  geometry: {geom_type}  |  batch size: {batch}")

    # ── Paginate through all features ─────────────────────────────────────────
    all_features = []
    offset = 0

    while True:
        print(f"  Fetching records {offset} – {offset + batch - 1} …", end=" ")
        data = query_layer(base_url, offset, batch)

        if "error" in data:
            sys.exit(f"\nQuery error: {data['error'].get('message', data['error'])}")

        features = data.get("features", [])
        all_features.extend(features)
        print(f"got {len(features)}")

        # Stop when fewer records than requested are returned
        if len(features) < batch:
            break

        # Some servers don't support pagination — check the flag
        if not data.get("exceededTransferLimit", False) and len(features) < batch:
            break

        offset += batch

    print(f"Total features fetched: {len(all_features)}")

    if not all_features:
        sys.exit("No features returned. Check the URL or layer permissions.")

    # ── Build CSV ─────────────────────────────────────────────────────────────
    # Collect all attribute keys (preserve order, lat/lon always first)
    attr_keys = []
    for feat in all_features:
        for k in feat.get("attributes", {}):
            if k not in attr_keys:
                attr_keys.append(k)

    fieldnames = ["latitude", "longitude"] + attr_keys

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()

        for feat in all_features:
            lat, lon = extract_coords(feat.get("geometry"))
            row = {"latitude": lat, "longitude": lon}
            row.update(feat.get("attributes", {}))
            writer.writerow(row)

    print(f"✓ Saved {len(all_features)} rows → {output_path}")


In [3]:
url = "https://services.arcgis.com/NuWFvHYDMVmmxMeM/ArcGIS/rest/services/NC_FatalAndSeriousInjuryCrashes/FeatureServer/0"
output = "/Users/root1/apps/ntsb/northcarolina/crashdata.csv"
main(url, output)

Connecting to: https://services.arcgis.com/NuWFvHYDMVmmxMeM/ArcGIS/rest/services/NC_FatalAndSeriousInjuryCrashes/FeatureServer/0
Layer: 'NC_FatalAndSeriousInjuryCrashes'  |  geometry: esriGeometryPoint  |  batch size: 1000
  Fetching records 0 – 999 … got 1000
  Fetching records 1000 – 1999 … got 1000
  Fetching records 2000 – 2999 … got 1000
  Fetching records 3000 – 3999 … got 1000
  Fetching records 4000 – 4999 … got 1000
  Fetching records 5000 – 5999 … got 1000
  Fetching records 6000 – 6999 … got 1000
  Fetching records 7000 – 7999 … got 1000
  Fetching records 8000 – 8999 … got 1000
  Fetching records 9000 – 9999 … got 1000
  Fetching records 10000 – 10999 … got 1000
  Fetching records 11000 – 11999 … got 1000
  Fetching records 12000 – 12999 … got 1000
  Fetching records 13000 – 13999 … got 1000
  Fetching records 14000 – 14999 … got 1000
  Fetching records 15000 – 15999 … got 1000
  Fetching records 16000 – 16999 … got 1000
  Fetching records 17000 – 17999 … got 1000
  Fetchin

In [9]:

pd.set_option('display.max_columns', None)
data

,latitude,longitude,FID,Crash_ID,GIS_RteTxt,GIS_Milepo,Longitude,Latitude,Source_ID,MapMethod,LOC_ERROR,Date,Time,Day_of_Wee,County_Nam,City,MPO_RPO_Na,Crash_Seve,Crash_Type,Num_Vehicl,Num_Non_Mo,Weather,Road_Condi,Light_Cond,Alcohol_Re,Speed_Rela,Unbelted_D,Motorcycle,Heavy_Truc,On_Road,Distance,Direction,From_Road,Toward_Roa,Crash_Repo
0,34.717884,-77.069970,1,104824194,30000058016,26.473,-77.069970,34.717884,0,RTEMP,NO ERROR,08/20/2016.,11:43,Saturday,Carteret,Rural,DOWN EAST RPO,A,Overturn/Rollover,3,0,Cloudy,Dry,Daylight,N,N,N,Y,N,NC 58,0.100,N,SR 1607,SR 1110,https://crashweb.ncdot.gov/crashweb/submitCras...
1,35.180171,-83.373604,2,104897542,40001729056,0.376,-83.373604,35.180171,0,RTEMP,NO ERROR,11/01/2016.,1:29,Tuesday,Macon,Franklin,SOUTHWESTERN RPO,K,Ran Off Road - Right,1,0,Clear,Dry,Dark - Lighted Roadway,Y,N,N,Y,N,DEPOT,0.000,,DEPOT,HARDWOOD,https://crashweb.ncdot.gov/crashweb/submitCras...
2,35.588722,-77.368320,3,106752041,21000264074,6.082,-77.368320,35.588722,0,RTEMP,NO ERROR,10/21/2021.,13:24,Thursday,Pitt,Greenville,GREENVILLE URBAN AREA MPO,A,"Sideswipe,Opposite Direction",2,0,Clear,Dry,Daylight,N,N,N,Y,N,NC 43,0.076,W,US 264ALT,BROOK,https://crashweb.ncdot.gov/crashweb/submitCras...
3,35.594905,-77.339114,4,107525703,21000264074,7.897,-77.339114,35.594905,0,RTEMP,NO ERROR,11/09/2023.,22:50,Thursday,Pitt,Greenville,GREENVILLE URBAN AREA MPO,A,"Right Turn, Different Roadways",2,0,Clear,Dry,Dark - Lighted Roadway,N,N,N,N,N,US 264ALT,0.000,,GOLDEN,ADAMS,https://crashweb.ncdot.gov/crashweb/submitCras...
4,35.098049,-78.968688,5,108101916,40001437026,1.216,-78.968688,35.098049,0,RTEMP,NO ERROR,04/12/2025.,15:09,Saturday,Cumberland,Fayetteville,FAYETTEVILLE AREA MPO,K,"Left Turn, Same Roadway",2,0,Clear,Dry,Daylight,N,Y,N,Y,N,SANTA FE,0.000,,ALL AMERICAN,COALITION,https://crashweb.ncdot.gov/crashweb/submitCras...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53912,36.504486,-79.388763,53913,108296115,30000086017,22.813,-79.388763,36.504486,21,XY,,11/01/2025.,2:29,Saturday,Caswell,Rural,PIEDMONT TRIAD RPO,A,"Sideswipe,Opposite Direction",2,0,Clear,Dry,Dark - Roadway Not Lighted,N,N,N,N,N,NC 86,0.700,N,SR 1300,SR 1500,https://crashweb.ncdot.gov/crashweb/submitCras...
53913,35.463062,-77.669012,53914,108296134,20000013040,9.736,-77.669012,35.463062,21,XY,,11/02/2025.,18:11,Sunday,Greene,Rural,EAST CAROLINA RPO,A,Angle,2,0,Clear,Dry,Dark - Roadway Not Lighted,N,N,N,N,N,US 258,0.000,,NC 91,SR 1400,https://crashweb.ncdot.gov/crashweb/submitCras...
53914,35.690057,-81.893604,53915,108296443,10000040059,24.259,-81.893604,35.690057,21,XY,,10/25/2025.,16:05,Saturday,Mcdowell,Rural,FOOTHILLS RPO,K,Pedestrian,1,1,Cloudy,Dry,Daylight,N,N,N,N,Y,I 40,0.100,E,*MILE 91,*MILE 92,https://crashweb.ncdot.gov/crashweb/submitCras...
53915,35.218669,-80.112331,53916,108295692,50024789084,999.999,-80.112331,35.218669,1,XY,,10/10/2025.,17:19,Friday,Stanly,Norwood,ROCKY RIVER RPO,A,Pedestrian,1,1,Clear,Dry,Dusk,N,N,N,N,N,PRICE,0.000,,FORK,*,https://crashweb.ncdot.gov/crashweb/submitCras...


In [2]:
data = pd.read_csv("crashdata.csv")

In [3]:
data.columns

Index(['latitude', 'longitude', 'FID', 'Crash_ID', 'GIS_RteTxt', 'GIS_Milepo',
       'Longitude', 'Latitude', 'Source_ID', 'MapMethod', 'LOC_ERROR', 'Date',
       'Time', 'Day_of_Wee', 'County_Nam', 'City', 'MPO_RPO_Na', 'Crash_Seve',
       'Crash_Type', 'Num_Vehicl', 'Num_Non_Mo', 'Weather', 'Road_Condi',
       'Light_Cond', 'Alcohol_Re', 'Speed_Rela', 'Unbelted_D', 'Motorcycle',
       'Heavy_Truc', 'On_Road', 'Distance', 'Direction', 'From_Road',
       'Toward_Roa', 'Crash_Repo'],
      dtype='str')

In [7]:
data[pd.isnull(data['Longitude'])]

,latitude,longitude,FID,Crash_ID,GIS_RteTxt,GIS_Milepo,Longitude,Latitude,Source_ID,MapMethod,...,Speed_Rela,Unbelted_D,Motorcycle,Heavy_Truc,On_Road,Distance,Direction,From_Road,Toward_Roa,Crash_Repo
